In [1]:
# ============================================================
# Dữ liệu WIND (NASA CDAWeb HAPI) đã được scrape sẵn bằng
# scrape_NASA_data.py — ở đây chỉ đọc lại file CSV, không gọi API.
# ============================================================

import pandas as pd

df = pd.read_csv("data/data_NASA/NASA_WIND_solar_wind.csv")

print("Data shape:", df.shape)
df.head()


Data shape: (5099, 6)


,Time,Proton_V_nonlin,Proton_Np_nonlin,BX,BY,BZ
0,2020-01-01T00:02:12.057Z,298.0,7.79,2.54,1.530,0.0369
1,2020-01-01T00:03:51.300Z,298.0,8.29,2.58,1.560,-0.0274
2,2020-01-01T00:05:30.544Z,293.0,5.77,2.81,1.480,-0.0857
3,2020-01-01T00:07:09.788Z,292.0,5.17,2.87,1.100,-0.0848
4,2020-01-01T00:08:45.930Z,292.0,5.28,2.91,0.869,0.1590


In [2]:
# ============================================================
# Loại bỏ dòng có giá trị lấp đầy (fill value)
# ============================================================

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
)

# Proton_V_nonlin có giá trị lấp đầy 99999.9 cho các dòng thiếu số
# liệu — không phải vận tốc thật, phải loại trước khi dùng dữ liệu.
wind_df = df[df["Proton_V_nonlin"] < 9999].reset_index(drop=True)
print(f"Loại bỏ {len(df) - len(wind_df)} dòng có giá trị lấp đầy (fill value).")
print(f"Số dòng còn lại: {len(wind_df)}")

Loại bỏ 9 dòng có giá trị lấp đầy (fill value).
Số dòng còn lại: 5090


In [3]:
import time

# ============================================================
# Genetic Programming cho Symbolic Regression — xem
# 04_symbolic_regression.ipynb để biết chi tiết thiết kế engine.
# ============================================================

FUNCTIONS_ARITY = {
    "+": 2, "-": 2, "*": 2, "/": 2, "sin": 1, "cos": 1,
}


def protected_div(a, b):
    with np.errstate(divide="ignore", invalid="ignore"):
        ket_qua = np.divide(a, b)
    return np.where(np.isfinite(ket_qua), ket_qua, 1.0)


def apply_op(op, args):
    if op == "+":
        return args[0] + args[1]
    if op == "-":
        return args[0] - args[1]
    if op == "*":
        return args[0] * args[1]
    if op == "/":
        return protected_div(args[0], args[1])
    if op == "sin":
        return np.sin(args[0])
    if op == "cos":
        return np.cos(args[0])
    raise ValueError(f"Toán tử không xác định: {op}")


class Node:
    __slots__ = ("op", "children")

    def __init__(self, op, children=None):
        self.op = op
        self.children = children if children is not None else []


def is_variable(op):
    return isinstance(op, str) and op.startswith("x") and op[1:].isdigit()


def eval_node(node, X):
    if is_variable(node.op):
        return X[:, int(node.op[1:])]
    if isinstance(node.op, (int, float)):
        return node.op
    args = [eval_node(con, X) for con in node.children]
    return apply_op(node.op, args)


def tree_depth(node):
    if not node.children:
        return 1
    return 1 + max(tree_depth(con) for con in node.children)


def tree_size(node):
    return 1 + sum(tree_size(con) for con in node.children)


def copy_tree(node):
    return Node(node.op, [copy_tree(con) for con in node.children])


def all_nodes(node):
    nodes = [node]
    for con in node.children:
        nodes.extend(all_nodes(con))
    return nodes


def tree_to_string(node, feature_names):
    if is_variable(node.op):
        return feature_names[int(node.op[1:])]
    if isinstance(node.op, (int, float)):
        return str(node.op)
    if len(node.children) == 1:
        return f"{node.op}({tree_to_string(node.children[0], feature_names)})"
    return (f"({tree_to_string(node.children[0], feature_names)} {node.op} "
            f"{tree_to_string(node.children[1], feature_names)})")


def random_terminal(rng, n_features):
    if rng.random() < 0.5:
        idx = rng.integers(0, n_features)
        return Node(f"x{idx}")
    return Node(round(float(rng.uniform(-2, 2)), 3))


def random_tree(rng, depth_remaining, method, n_features):
    la_la = depth_remaining <= 1 or (method == "grow" and rng.random() < 0.3)
    if la_la:
        return random_terminal(rng, n_features)
    op = rng.choice(list(FUNCTIONS_ARITY.keys()))
    arity = FUNCTIONS_ARITY[op]
    return Node(op, [random_tree(rng, depth_remaining - 1, method, n_features) for _ in range(arity)])


def init_population(rng, population_size, min_depth, max_depth, n_features):
    population = []
    do_sau_list = list(range(min_depth, max_depth + 1))
    for i in range(population_size):
        do_sau = do_sau_list[i % len(do_sau_list)]
        method = "full" if i % 2 == 0 else "grow"
        population.append(random_tree(rng, do_sau, method, n_features))
    return population


def crossover(rng, parent1, parent2, max_depth):
    child1 = copy_tree(parent1)
    child2 = copy_tree(parent2)

    nodes1 = all_nodes(child1)
    nodes2 = all_nodes(child2)
    diem1 = nodes1[rng.integers(0, len(nodes1))]
    diem2 = nodes2[rng.integers(0, len(nodes2))]

    diem1.op, diem2.op = diem2.op, diem1.op
    diem1.children, diem2.children = diem2.children, diem1.children

    if tree_depth(child1) > max_depth or tree_depth(child2) > max_depth:
        return copy_tree(parent1), copy_tree(parent2)

    return child1, child2


def mutate(rng, individual, max_depth, n_features):
    child = copy_tree(individual)
    nodes = all_nodes(child)
    diem = nodes[rng.integers(0, len(nodes))]

    cay_moi = random_tree(rng, int(rng.integers(1, 4)), rng.choice(["grow", "full"]), n_features)
    diem.op = cay_moi.op
    diem.children = cay_moi.children

    if tree_depth(child) > max_depth:
        return copy_tree(individual)

    return child


def tournament_selection(rng, population, fitnesses, tournament_size):
    indices = rng.integers(0, len(population), size=tournament_size)
    best_index = indices[np.argmin(fitnesses[indices])]
    return population[best_index]


PARSIMONY_COEF = 0.0001


def fitness(node, X, y):
    y_pred = eval_node(node, X)
    y_pred = np.broadcast_to(np.asarray(y_pred, dtype=float), (X.shape[0],))

    if not np.all(np.isfinite(y_pred)):
        return np.inf

    mse = np.mean((y_pred - y) ** 2)

    return mse + PARSIMONY_COEF * tree_size(node)


CROSSOVER_RATE = 0.8
MUTATION_RATE = 0.15
ELITE_SIZE = 2
TOURNAMENT_SIZE = 3


def genetic_programming(
    X, y,
    population_size,
    generations,
    max_depth=6,

    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,
    elite_size=ELITE_SIZE,
    tournament_size=TOURNAMENT_SIZE,

    seed=42,
):
    rng = np.random.default_rng(seed)
    n_features = X.shape[1]

    population = init_population(rng, population_size, min_depth=2, max_depth=max_depth, n_features=n_features)
    fitnesses = np.array([fitness(ind, X, y) for ind in population])

    best_index = int(np.argmin(fitnesses))
    best_tree = copy_tree(population[best_index])
    best_fitness = fitnesses[best_index]

    history = []

    start_time = time.perf_counter()

    for generation in range(generations):

        current_best = int(np.argmin(fitnesses))
        if fitnesses[current_best] < best_fitness:
            best_fitness = fitnesses[current_best]
            best_tree = copy_tree(population[current_best])

        history.append(best_fitness)

        order = np.argsort(fitnesses)
        new_population = [copy_tree(population[i]) for i in order[:elite_size]]

        while len(new_population) < population_size:
            parent1 = tournament_selection(rng, population, fitnesses, tournament_size)
            parent2 = tournament_selection(rng, population, fitnesses, tournament_size)

            if rng.random() < crossover_rate:
                child1, child2 = crossover(rng, parent1, parent2, max_depth)
            else:
                child1, child2 = copy_tree(parent1), copy_tree(parent2)

            if rng.random() < mutation_rate:
                child1 = mutate(rng, child1, max_depth, n_features)
            if rng.random() < mutation_rate:
                child2 = mutate(rng, child2, max_depth, n_features)

            new_population.append(child1)
            if len(new_population) < population_size:
                new_population.append(child2)

        population = new_population
        fitnesses = np.array([fitness(ind, X, y) for ind in population])

    current_best = int(np.argmin(fitnesses))
    if fitnesses[current_best] < best_fitness:
        best_fitness = fitnesses[current_best]
        best_tree = copy_tree(population[current_best])

    elapsed_time = time.perf_counter() - start_time

    generations_run = int(np.argmin(history)) + 1

    return {
        "tree": best_tree,
        "fitness": best_fitness,
        "time": elapsed_time,
        "history": history,
        "generations": generations,
        "generations_run": generations_run,
        "seed": seed,
    }

In [4]:
# ============================================================
# SYMBOLIC REGRESSION — Bình phương độ lớn từ trường
# |B|^2 = BX^2 + BY^2 + BZ^2 — quan hệ TỔNG BÌNH PHƯƠNG (Pythagore),
# không phải dạng lũy thừa nên log-transform KHÔNG biến nó thành tuyến
# tính được (BX/BY/BZ còn có thể âm nên không log được nữa). Linear
# Regression trên BX, BY, BZ thô (không có số hạng bình phương) do đó
# gần như chắc chắn thất bại — đây là phép thử công bằng cho lợi thế
# cấu trúc thật của GP (không nhờ mẹo log-space nào cả).
# ============================================================

wind_df["B_sq"] = wind_df["BX"]**2 + wind_df["BY"]**2 + wind_df["BZ"]**2

B_FEATURE_COLS = ["BX", "BY", "BZ"]
B_TARGET_COL = "B_sq"

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    wind_df[B_FEATURE_COLS], wind_df[B_TARGET_COL],
    test_size=0.2, random_state=42,
)

# KHÔNG log-transform (BX/BY/BZ có dấu âm) — chỉ chuẩn hóa.
scaler_b = StandardScaler()
Xb_train_scaled = scaler_b.fit_transform(Xb_train)
Xb_test_scaled = scaler_b.transform(Xb_test)

print(f"Train: {len(Xb_train)} dòng")
print(f"Test : {len(Xb_test)} dòng")

Train: 4072 dòng
Test : 1018 dòng


In [5]:
model_b = LinearRegression()
model_b.fit(Xb_train_scaled, yb_train)

y_pred_b = model_b.predict(Xb_test_scaled)

mae_b = mean_absolute_error(yb_test, y_pred_b)
rmse_b = mean_squared_error(yb_test, y_pred_b) ** 0.5
r2_b = r2_score(yb_test, y_pred_b)

print(f"MAE : {mae_b:.4f}")
print(f"RMSE: {rmse_b:.4f}")
print(f"R2  : {r2_b:.4f}")

MAE : 15.1284
RMSE: 19.9825
R2  : 0.4784


In [6]:
# Phương sai target (B_sq) ~790, lớn hơn hẳn airfoil (~46.9, tỉ lệ
# ~16.8x) — tăng PARSIMONY_COEF theo tỉ lệ đó.
PARSIMONY_COEF = 0.3

GP_POPULATION_SIZE = 500
GP_GENERATIONS = 500
GP_MAX_DEPTH = 7

yb_train_arr = yb_train.to_numpy()
yb_test_arr = yb_test.to_numpy()

gp_result_b = genetic_programming(
    Xb_train_scaled, yb_train_arr,
    population_size=GP_POPULATION_SIZE,
    generations=GP_GENERATIONS,
    max_depth=GP_MAX_DEPTH,
    seed=42,
)

print(f"Thời gian chạy GP: {gp_result_b['time']:.2f}s")
print(f"Fitness tốt nhất (MSE + phạt kích thước): {gp_result_b['fitness']:.4f}")
print(f"Đạt được từ thế hệ: {gp_result_b['generations_run']}/{gp_result_b['generations']}")
print(f"Kích thước cây: {tree_size(gp_result_b['tree'])} nút, độ sâu {tree_depth(gp_result_b['tree'])}")
print(f"\nBiểu thức GP tìm được (dự đoán B_sq):\n{tree_to_string(gp_result_b['tree'], B_FEATURE_COLS)}")

Thời gian chạy GP: 308.17s
Fitness tốt nhất (MSE + phạt kích thước): 52.2041
Đạt được từ thế hệ: 332/500
Kích thước cây: 47 nút, độ sâu 7

Biểu thức GP tìm được (dự đoán B_sq):
((((((BZ + BX) * BZ) - ((-0.848 + BY) / (-1.502 * -0.128))) - (((-0.832 + BY) + (BY + BX)) * (-0.744 + (-0.332 - BY)))) * (-1.595 / sin(sin(sin(-0.332))))) * cos((((cos(BZ) - -1.492) - sin(BX)) / (-1.281 / -0.332))))


In [7]:
y_pred_gp_b = eval_node(gp_result_b["tree"], Xb_test_scaled)
y_pred_gp_b = np.broadcast_to(np.asarray(y_pred_gp_b, dtype=float), (Xb_test_scaled.shape[0],))

mae_gp_b = mean_absolute_error(yb_test_arr, y_pred_gp_b)
rmse_gp_b = mean_squared_error(yb_test_arr, y_pred_gp_b) ** 0.5
r2_gp_b = r2_score(yb_test_arr, y_pred_gp_b)

print(f"MAE : {mae_gp_b:.4f}")
print(f"RMSE: {rmse_gp_b:.4f}")
print(f"R2  : {r2_gp_b:.4f}")

MAE : 4.1820
RMSE: 6.0779
R2  : 0.9517


In [8]:
pd.DataFrame([
    {"Mô hình": "Linear Regression", "MAE": mae_b, "RMSE": rmse_b, "R2": r2_b},
    {"Mô hình": "GP (Symbolic Regression)", "MAE": mae_gp_b, "RMSE": rmse_gp_b, "R2": r2_gp_b},
]).round(4)

,Mô hình,MAE,RMSE,R2
0,Linear Regression,15.1284,19.9825,0.4784
1,GP (Symbolic Regression),4.1820,6.0779,0.9517


In [9]:
# ============================================================
# SYMBOLIC REGRESSION — Vận tốc Alfvén bình phương
# V_A^2 ∝ |B|^2 / n = (BX^2+BY^2+BZ^2) / Proton_Np_nonlin — kết hợp CẢ
# HAI dạng phi tuyến trong một bài toán: tổng bình phương (như |B|^2 ở
# trên) VÀ phép chia cho mật độ hạt n. Đây là đại lượng vật lý plasma
# quan trọng (tốc độ lan truyền nhiễu loạn dọc theo đường sức từ).
# Linear Regression trên BX, BY, BZ, Np thô (không có số hạng bình
# phương lẫn phép chia) khó hơn cả trường hợp |B|^2; GP vừa phải tự
# tạo ra bình phương, vừa phải tìm đúng phép chia cho biến mật độ.
# ============================================================

wind_df["VA_sq"] = (wind_df["BX"]**2 + wind_df["BY"]**2 + wind_df["BZ"]**2) / wind_df["Proton_Np_nonlin"]

VA_FEATURE_COLS = ["BX", "BY", "BZ", "Proton_Np_nonlin"]
VA_TARGET_COL = "VA_sq"

Xva_train, Xva_test, yva_train, yva_test = train_test_split(
    wind_df[VA_FEATURE_COLS], wind_df[VA_TARGET_COL],
    test_size=0.2, random_state=42,
)

# KHÔNG log-transform (BX/BY/BZ có dấu âm) — chỉ chuẩn hóa.
scaler_va = StandardScaler()
Xva_train_scaled = scaler_va.fit_transform(Xva_train)
Xva_test_scaled = scaler_va.transform(Xva_test)

print(f"Train: {len(Xva_train)} dòng")
print(f"Test : {len(Xva_test)} dòng")


Train: 4072 dòng
Test : 1018 dòng


In [10]:
model_va = LinearRegression()
model_va.fit(Xva_train_scaled, yva_train)

y_pred_va = model_va.predict(Xva_test_scaled)

mae_va = mean_absolute_error(yva_test, y_pred_va)
rmse_va = mean_squared_error(yva_test, y_pred_va) ** 0.5
r2_va = r2_score(yva_test, y_pred_va)

print(f"MAE : {mae_va:.4f}")
print(f"RMSE: {rmse_va:.4f}")
print(f"R2  : {r2_va:.4f}")


MAE : 2.7376
RMSE: 3.6804
R2  : 0.4629


In [11]:
# Phương sai target (VA_sq) ~25.2, nhỏ hơn airfoil (~46.9, tỉ lệ
# ~0.54x) nhưng bài toán khó hơn về cấu trúc (cần cả bình phương lẫn
# phép chia đúng biến) — giảm PARSIMONY_COEF theo tỉ lệ phương sai
# nhưng không giảm quá mức để tránh cây quá đơn giản.
PARSIMONY_COEF = 0.01

GP_POPULATION_SIZE = 500
GP_GENERATIONS = 500
GP_MAX_DEPTH = 7

yva_train_arr = yva_train.to_numpy()
yva_test_arr = yva_test.to_numpy()

gp_result_va = genetic_programming(
    Xva_train_scaled, yva_train_arr,
    population_size=GP_POPULATION_SIZE,
    generations=GP_GENERATIONS,
    max_depth=GP_MAX_DEPTH,
    seed=42,
)

print(f"Thời gian chạy GP: {gp_result_va['time']:.2f}s")
print(f"Fitness tốt nhất (MSE + phạt kích thước): {gp_result_va['fitness']:.4f}")
print(f"Đạt được từ thế hệ: {gp_result_va['generations_run']}/{gp_result_va['generations']}")
print(f"Kích thước cây: {tree_size(gp_result_va['tree'])} nút, độ sâu {tree_depth(gp_result_va['tree'])}")
print(f"\nBiểu thức GP tìm được (dự đoán VA_sq):\n{tree_to_string(gp_result_va['tree'], VA_FEATURE_COLS)}")


Thời gian chạy GP: 282.56s
Fitness tốt nhất (MSE + phạt kích thước): 3.1747
Đạt được từ thế hệ: 476/500
Kích thước cây: 55 nút, độ sâu 7

Biểu thức GP tìm được (dự đoán VA_sq):
((((BX * (1.023 - -0.479)) * BZ) - (cos(((BX - Proton_Np_nonlin) - 0.737)) - sin((cos(Proton_Np_nonlin) - (-0.665 + BY))))) - ((((BY * BY) - ((-0.665 + BY) + Proton_Np_nonlin)) * Proton_Np_nonlin) - ((((BY / -0.496) * (1.023 - BY)) - (-1.997 + cos(BZ))) - (Proton_Np_nonlin - ((BX * 1.36) - Proton_Np_nonlin)))))


In [12]:
y_pred_gp_va = eval_node(gp_result_va["tree"], Xva_test_scaled)
y_pred_gp_va = np.broadcast_to(np.asarray(y_pred_gp_va, dtype=float), (Xva_test_scaled.shape[0],))

mae_gp_va = mean_absolute_error(yva_test_arr, y_pred_gp_va)
rmse_gp_va = mean_squared_error(yva_test_arr, y_pred_gp_va) ** 0.5
r2_gp_va = r2_score(yva_test_arr, y_pred_gp_va)

print(f"MAE : {mae_gp_va:.4f}")
print(f"RMSE: {rmse_gp_va:.4f}")
print(f"R2  : {r2_gp_va:.4f}")


MAE : 1.0554
RMSE: 1.5486
R2  : 0.9049


In [13]:
pd.DataFrame([
    {"Mô hình": "Linear Regression", "MAE": mae_va, "RMSE": rmse_va, "R2": r2_va},
    {"Mô hình": "GP (Symbolic Regression)", "MAE": mae_gp_va, "RMSE": rmse_gp_va, "R2": r2_gp_va},
]).round(4)


,Mô hình,MAE,RMSE,R2
0,Linear Regression,2.7376,3.6804,0.4629
1,GP (Symbolic Regression),1.0554,1.5486,0.9049
